# Módulo 04 · Aula 02 — Iteradores e Geradores

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Rodei o relatório no arquivo do ano inteiro e o computador travou. O CSV tem 4 GB e o script tentou carregar tudo na memória de uma vez."*

Você já viu `for linha in arquivo:` e `range(1_000_000)` funcionarem sem estourar a RAM. Esta aula explica **por quê** — e ensina você a construir suas próprias estruturas com essa propriedade.

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | O protocolo de iteração | O que o `for` realmente faz |
| 2 | Iterável vs iterador | A distinção que confunde todo mundo |
| 3 | Iteradores próprios (`__iter__`/`__next__`) | Objetos que se percorrem |
| 4 | **Geradores e `yield`** | Iteradores em 3 linhas |
| 5 | Avaliação preguiçosa | Processar 4 GB com 10 MB de RAM |
| 6 | `yield from` | Compor geradores |
| 7 | Geradores bidirecionais (`send`) | Corrotinas |
| 8 | `itertools` | O kit de iteração da biblioteca padrão |
| 9 | Pipelines | Encadear transformações |

## 1. O que o `for` realmente faz

Quando você escreve:

```python
for item in colecao:
    print(item)
```

Python executa, nos bastidores:

```python
iterador = iter(colecao)     # 1. pede um iterador
while True:
    try:
        item = next(iterador)  # 2. pede o próximo
    except StopIteration:      # 3. até acabar
        break
    print(item)
```

Duas funções embutidas e uma exceção. **É só isso.**

In [ ]:
cidades = ["Campinas", "São Paulo", "Sorocaba"]

# O for, desmontado
iterador = iter(cidades)
print("iterador:", iterador)
print(next(iterador))
print(next(iterador))
print(next(iterador))

try:
    next(iterador)
except StopIteration:
    print("StopIteration — acabou")

## 2. Iterável ≠ Iterador

Esta distinção é a fonte de metade da confusão sobre o assunto.

| | **Iterável** | **Iterador** |
|---|-------------|--------------|
| Define | `__iter__()` | `__iter__()` **e** `__next__()` |
| É | Uma coleção que **pode** ser percorrida | O objeto que **está** percorrendo |
| Tem estado? | Não | **Sim** — lembra onde parou |
| Reutilizável? | ✅ Sim, quantas vezes quiser | ❌ Não, esgota |
| Exemplos | `list`, `dict`, `str`, `set`, arquivo | resultado de `iter()`, `map`, `filter`, gerador |

**Analogia:** a lista é o **livro**; o iterador é o **marcador de página**. Você pode ter vários marcadores no mesmo livro, cada um numa página diferente. Quando o marcador chega ao fim, ele não volta sozinho — você pega um novo.

In [ ]:
lista = [1, 2, 3]

# Iterável: percorrer não o consome
print("1ª passada:", [x for x in lista])
print("2ª passada:", [x for x in lista])

# Iterador: esgota
it = iter(lista)
print("\n1ª passada do iterador:", list(it))
print("2ª passada do iterador:", list(it), "  ← 😱 vazio")

In [ ]:
# Dois iteradores independentes sobre a MESMA lista
a = iter(lista)
b = iter(lista)

print("a:", next(a), "| b:", next(b))
print("a:", next(a), "| b:", next(b))
print("\nSão o mesmo objeto?", a is b)

In [ ]:
# Como saber o que é o quê
from collections.abc import Iterable, Iterator

exemplos = {
    "lista":        [1, 2, 3],
    "tupla":        (1, 2, 3),
    "string":       "abc",
    "dict":         {"a": 1},
    "set":          {1, 2},
    "range":        range(3),
    "iter(lista)":  iter([1, 2, 3]),
    "map":          map(str, [1, 2]),
    "gerador":      (x for x in range(3)),
    "int":          42,
}

print(f"{'objeto':<14}{'Iterable':>10}{'Iterator':>10}")
print("─" * 34)
for nome, obj in exemplos.items():
    print(f"{nome:<14}{str(isinstance(obj, Iterable)):>10}{str(isinstance(obj, Iterator)):>10}")

> 💡 **Todo iterador é iterável, mas nem todo iterável é iterador.** O `__iter__` de um iterador devolve `self` — é isso que permite usar um iterador direto no `for`.
>
> 🔴 **A consequência prática:** `map`, `filter`, `zip`, `enumerate`, `reversed` e geradores são **iteradores**. Se você consumir o resultado uma vez, ele acaba. Este é um bug real e silencioso:
> ```python
> dados = map(processar, linhas)
> print(f"{len(list(dados))} registros")   # consome tudo
> for d in dados:                          # laço nunca executa!
>     ...
> ```

## 3. Escrevendo um iterador próprio

Para que um objeto funcione no `for`, ele precisa do **protocolo**:

- `__iter__()` → devolve um iterador (frequentemente `self`)
- `__next__()` → devolve o próximo item, ou levanta `StopIteration`

In [ ]:
class ContadorRegressivo:
    """Um iterador feito à mão, para você ver o mecanismo."""

    def __init__(self, inicio):
        self.inicio = inicio

    def __iter__(self):
        # Devolve um ITERADOR NOVO a cada chamada.
        # Isso torna a classe reutilizável em vários for.
        self.atual = self.inicio
        return self

    def __next__(self):
        if self.atual <= 0:
            raise StopIteration
        self.atual -= 1
        return self.atual + 1


for n in ContadorRegressivo(5):
    print(n, end=" ")
print("💥")

In [ ]:
# A mesma coisa como GERADOR: 3 linhas em vez de 12
def contador_regressivo(inicio):
    while inicio > 0:
        yield inicio
        inicio -= 1


for n in contador_regressivo(5):
    print(n, end=" ")
print("💥")

> 💭 **Compare os dois blocos.** Mesma funcionalidade, 12 linhas contra 3. Não há nenhuma vantagem na versão de classe para este caso.
>
> **Regra prática:** escreva geradores. Só use a classe quando precisar de outros métodos além da iteração (`reset()`, `progresso()`, um `__len__`...).

## 4. Geradores e `yield`

Uma função que contém `yield` **não é uma função comum**. Ela é uma **função geradora**: chamá-la não executa nada — devolve um objeto gerador.

**O que o `yield` faz:**

1. Devolve o valor
2. **Congela** a função inteira: variáveis locais, posição da execução, tudo
3. No próximo `next()`, **descongela** e continua exatamente de onde parou

In [ ]:
def demonstrar():
    print("  [1] início da função")
    yield "primeiro"
    print("  [2] depois do primeiro yield")
    yield "segundo"
    print("  [3] depois do segundo yield")
    yield "terceiro"
    print("  [4] fim da função")


print("Chamando a função geradora:")
g = demonstrar()
print("  resultado:", g, "\n  ← nada foi executado ainda!\n")

print("next() #1:")
print("  →", next(g), "\n")
print("next() #2:")
print("  →", next(g), "\n")
print("next() #3:")
print("  →", next(g), "\n")
print("next() #4:")
try:
    next(g)
except StopIteration:
    print("  StopIteration")

### `return` vs `yield`

| | `return` | `yield` |
|---|----------|---------|
| Devolve | Um valor, uma vez | Vários valores, sob demanda |
| Ao devolver | **Encerra** a função | **Pausa** a função |
| Estado local | Perdido | Preservado |
| Memória | Todo o resultado de uma vez | Um item por vez |

In [ ]:
import sys

def com_return(n):
    resultado = []
    for i in range(n):
        resultado.append(i ** 2)
    return resultado          # constrói a lista INTEIRA


def com_yield(n):
    for i in range(n):
        yield i ** 2          # produz sob demanda


n = 1_000_000
lista = com_return(n)
gerador = com_yield(n)

print(f"Lista   : {sys.getsizeof(lista):>12,} bytes")
print(f"Gerador : {sys.getsizeof(gerador):>12,} bytes")
print(f"Razão   : {sys.getsizeof(lista) / sys.getsizeof(gerador):>12,.0f}x")
print("\nMas o gerador não 'contém' os valores — ele sabe como produzi-los.")

### Geradores infinitos

Como o gerador só produz sob demanda, ele pode ser **infinito**. Você consome o quanto precisar.

In [ ]:
def numeros_pedido(inicio=1000):
    """Sequência infinita de números de pedido."""
    n = inicio
    while True:
        yield n
        n += 1


gerador = numeros_pedido()
print([next(gerador) for _ in range(5)])
print("O gerador continua vivo, pronto para o próximo:", next(gerador))

In [ ]:
from itertools import islice

def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


# islice corta um iterador sem materializá-lo
print("Primeiros 12:", list(islice(fibonacci(), 12)))

# Consumindo até uma condição
grandes = (n for n in fibonacci() if n > 1000)
print("1º Fibonacci acima de 1000:", next(grandes))

> ⚠️ **`list(fibonacci())` trava o kernel.** Um gerador infinito precisa de uma condição de parada: `islice`, `break`, `next()`, `takewhile`. Se você rodar sem querer, use o botão ⏹ **Interrupt** do VS Code.

## 5. Avaliação preguiçosa na prática

Aqui está a resposta para a dor do início da aula.

In [ ]:
from pathlib import Path
import csv
import random

# Criamos um CSV grandinho para experimentar
Path("dados_aula04").mkdir(exist_ok=True)
CSV = Path("dados_aula04/vendas_grande.csv")

rnd = random.Random(42)
cidades = ["Campinas", "São Paulo", "Sorocaba", "Curitiba", "Salvador"]
produtos = ["Notebook", "Mouse", "Monitor", "Teclado", "SSD"]

with open(CSV, "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(["id", "cidade", "produto", "quantidade", "preco", "status"])
    for i in range(1, 200_001):
        escritor.writerow([
            i, rnd.choice(cidades), rnd.choice(produtos),
            rnd.randint(1, 10), round(rnd.uniform(50, 3000), 2),
            rnd.choices(["pago", "pendente", "cancelado"], weights=[80, 12, 8])[0],
        ])

print(f"✅ {CSV} criado: {CSV.stat().st_size / 1024 / 1024:.1f} MB, 200.000 linhas")

In [ ]:
import tracemalloc
import time

# ❌ ABORDAGEM GULOSA: carrega tudo na memória
def ler_tudo(caminho):
    with open(caminho, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))     # a lista INTEIRA


tracemalloc.start()
inicio = time.perf_counter()

registros = ler_tudo(CSV)
total = sum(int(r["quantidade"]) * float(r["preco"])
            for r in registros if r["status"] == "pago")

tempo = time.perf_counter() - inicio
_, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"GULOSA    | {len(registros):,} registros")
print(f"          | faturamento: R$ {total:,.2f}")
print(f"          | tempo: {tempo:.2f}s | pico de memória: {pico/1024/1024:.1f} MB")

In [ ]:
# ✅ ABORDAGEM PREGUIÇOSA: uma linha por vez
def ler_preguicoso(caminho):
    """Gerador: nunca tem mais de uma linha na memória."""
    with open(caminho, newline="", encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            yield linha


tracemalloc.start()
inicio = time.perf_counter()

total = sum(int(r["quantidade"]) * float(r["preco"])
            for r in ler_preguicoso(CSV) if r["status"] == "pago")

tempo = time.perf_counter() - inicio
_, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"PREGUIÇOSA| faturamento: R$ {total:,.2f}")
print(f"          | tempo: {tempo:.2f}s | pico de memória: {pico/1024/1024:.1f} MB")

> 💭 **O mesmo resultado, com uma fração da memória.** E o mais importante: o consumo do gerador é **constante**. Com 200 mil linhas ou 200 milhões, o pico é o mesmo — porque nunca há mais de uma linha viva.
>
> É assim que se processa um arquivo maior que a RAM da máquina.
>
> ⚠️ **O preço:** você só passa uma vez. Se precisar de duas passadas (uma para a média, outra para o desvio), ou você materializa, ou lê o arquivo duas vezes, ou usa um algoritmo de passada única.

### O padrão pipeline

Geradores encadeados formam uma esteira. Cada etapa processa um item e passa adiante — **nenhuma lista intermediária é criada**.

```
arquivo → [ler] → [validar] → [filtrar] → [transformar] → resultado
           gen       gen         gen           gen
```

In [ ]:
def ler_csv(caminho):
    """Etapa 1: lê linhas cruas."""
    with open(caminho, newline="", encoding="utf-8") as f:
        yield from csv.DictReader(f)


def converter_tipos(linhas):
    """Etapa 2: converte, descartando linhas ruins."""
    for linha in linhas:
        try:
            yield {
                "id": int(linha["id"]),
                "cidade": linha["cidade"],
                "produto": linha["produto"],
                "quantidade": int(linha["quantidade"]),
                "preco": float(linha["preco"]),
                "status": linha["status"],
            }
        except (ValueError, KeyError):
            continue


def apenas_pagos(registros):
    """Etapa 3: filtra."""
    for r in registros:
        if r["status"] == "pago":
            yield r


def calcular_total(registros):
    """Etapa 4: enriquece."""
    for r in registros:
        r["total"] = round(r["quantidade"] * r["preco"], 2)
        yield r


# Montando o pipeline — nada foi executado ainda!
pipeline = calcular_total(apenas_pagos(converter_tipos(ler_csv(CSV))))
print("Pipeline montado:", pipeline)
print("Nenhuma linha lida até agora.\n")

# Só agora o arquivo começa a ser lido
print("Primeiros 3 registros:")
for r in islice(pipeline, 3):
    print("  ", r)

In [ ]:
# Agregando o pipeline inteiro com memória constante
from collections import defaultdict

tracemalloc.start()

por_cidade = defaultdict(float)
for r in calcular_total(apenas_pagos(converter_tipos(ler_csv(CSV)))):
    por_cidade[r["cidade"]] += r["total"]

_, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Pico de memória: {pico/1024/1024:.1f} MB\n")
for cidade, total in sorted(por_cidade.items(), key=lambda kv: -kv[1]):
    print(f"  {cidade:<14} R$ {total:>14,.2f}")

## 6. `yield from` — delegando a outro iterável

`yield from iteravel` é açúcar para `for x in iteravel: yield x`.

Mas ele faz mais: repassa também `send()`, `throw()` e o valor de retorno do subgerador.

In [ ]:
def sem_yield_from(colecoes):
    for colecao in colecoes:
        for item in colecao:
            yield item


def com_yield_from(colecoes):
    for colecao in colecoes:
        yield from colecao


dados = [[1, 2], [3, 4], [5, 6]]
print(list(sem_yield_from(dados)))
print(list(com_yield_from(dados)))

In [ ]:
# Achatando estruturas de profundidade arbitrária — recursão + yield from
def achatar(estrutura):
    """Achata listas aninhadas em qualquer profundidade."""
    for item in estrutura:
        if isinstance(item, (list, tuple)):
            yield from achatar(item)      # delega a si mesmo
        else:
            yield item


aninhado = [1, [2, 3, [4, 5, [6, 7]], 8], 9, [[10]]]
print(list(achatar(aninhado)))

In [ ]:
# Concatenando fontes sem carregar nada
def todos_os_arquivos(*caminhos):
    """Percorre vários CSVs como se fossem um só."""
    for caminho in caminhos:
        yield from ler_csv(caminho)


# (usando o mesmo arquivo duas vezes só para demonstrar)
print("Total de linhas em 2 arquivos:", sum(1 for _ in todos_os_arquivos(CSV, CSV)))

## 7. Geradores bidirecionais

Além de **produzir**, um gerador pode **receber** valores com `send()`. Isso o transforma numa **corrotina** — a base conceitual do `asyncio` (aula 04_06).

| Método | Faz |
|--------|-----|
| `next(g)` | Avança até o próximo `yield` |
| `g.send(valor)` | Avança **enviando** um valor para o `yield` |
| `g.throw(Exc)` | Levanta uma exceção dentro do gerador |
| `g.close()` | Encerra (levanta `GeneratorExit` lá dentro) |

In [ ]:
def acumulador():
    """Recebe valores e devolve o total acumulado."""
    total = 0
    while True:
        valor = yield total        # o yield DEVOLVE e RECEBE
        if valor is not None:
            total += valor


acc = acumulador()
next(acc)                          # ⚠️ "priming": avança até o 1º yield

print(acc.send(100))
print(acc.send(50))
print(acc.send(25))
acc.close()

> ⚠️ **O "priming" é obrigatório.** Antes do primeiro `send()`, o gerador precisa estar parado num `yield`. Chamar `send(valor)` num gerador recém-criado levanta `TypeError`. Por isso o `next(acc)` inicial.
>
> 💭 **Corrotinas foram a origem do `asyncio`.** Antes do `async`/`await` (Python 3.5), a programação assíncrona era feita com geradores e `yield from`. Entender isto ajuda muito na aula 04_06.

In [ ]:
def coletor_com_estatisticas():
    """Corrotina que acumula e reporta estatísticas ao ser fechada."""
    valores = []
    try:
        while True:
            valor = yield
            valores.append(valor)
    except GeneratorExit:
        if valores:
            print(f"\n📊 Fechando: {len(valores)} valores")
            print(f"   soma  : {sum(valores):,.2f}")
            print(f"   média : {sum(valores)/len(valores):,.2f}")
            print(f"   máximo: {max(valores):,.2f}")


coletor = coletor_com_estatisticas()
next(coletor)
for v in [1200.50, 890.00, 3400.75, 250.00]:
    coletor.send(v)
coletor.close()

## 8. `itertools` — o kit de iteração

A biblioteca padrão traz dezenas de ferramentas para iteradores. Estas são as que valem a pena decorar:

### Infinitos

| Função | Produz |
|--------|--------|
| `count(inicio, passo)` | `10, 20, 30, ...` para sempre |
| `cycle(iteravel)` | Repete a coleção infinitamente |
| `repeat(valor, n)` | O mesmo valor n vezes |

### Corte e filtro

| Função | Faz |
|--------|-----|
| `islice(it, fim)` | Fatia sem materializar |
| `takewhile(pred, it)` | Enquanto a condição for verdadeira |
| `dropwhile(pred, it)` | Descarta até a condição falhar |
| `filterfalse(pred, it)` | O inverso do `filter` |
| `compress(it, mascara)` | Filtra por uma máscara de booleanos |

### Combinação

| Função | Faz |
|--------|-----|
| `chain(*its)` | Concatena vários iteráveis |
| `zip_longest(a, b, fillvalue=)` | `zip` que vai até o mais **longo** |
| `product(a, b)` | Produto cartesiano |
| `combinations(it, n)` | Combinações sem repetição |
| `permutations(it, n)` | Permutações |

### Agrupamento

| Função | Faz |
|--------|-----|
| `groupby(it, key)` | Agrupa **consecutivos** ⚠️ |
| `accumulate(it, f)` | Soma (ou outra operação) acumulada |
| `pairwise(it)` | Pares consecutivos (3.10+) |
| `tee(it, n)` | Clona um iterador em n cópias |

In [ ]:
from itertools import (count, cycle, islice, takewhile, dropwhile, chain,
                       zip_longest, groupby, accumulate, pairwise, tee,
                       combinations, product)

print("count(100, 25) :", list(islice(count(100, 25), 5)))
print("cycle           :", list(islice(cycle(["site", "app", "mkt"]), 7)))
print("takewhile <500  :", list(takewhile(lambda x: x < 500, [100, 250, 400, 900, 200])))
print("dropwhile <500  :", list(dropwhile(lambda x: x < 500, [100, 250, 400, 900, 200])))
print("chain           :", list(chain([1, 2], [3, 4], [5])))
print("zip_longest     :", list(zip_longest([1, 2, 3], "ab", fillvalue="—")))
print("accumulate      :", list(accumulate([100, 250, 400, 900])))
print("pairwise        :", list(pairwise([1, 2, 3, 4])))
print("combinations    :", list(combinations("ABC", 2)))

### ⚠️ `groupby` só agrupa **consecutivos**

Esta é a pegadinha nº 1 do `itertools`. Diferente do `GROUP BY` do SQL, ele **não** reúne itens espalhados — só sequências adjacentes.

**Você precisa ordenar antes, pela mesma chave.**

In [ ]:
vendas = [
    {"cidade": "Campinas",  "valor": 1200},
    {"cidade": "São Paulo", "valor": 890},
    {"cidade": "Campinas",  "valor": 3400},
    {"cidade": "São Paulo", "valor": 250},
    {"cidade": "Campinas",  "valor": 780},
]

print("❌ SEM ordenar antes:")
for cidade, grupo in groupby(vendas, key=lambda v: v["cidade"]):
    itens = list(grupo)
    print(f"   {cidade:<12} {len(itens)} venda(s)")

print("\n✅ ORDENANDO pela mesma chave primeiro:")
ordenado = sorted(vendas, key=lambda v: v["cidade"])
for cidade, grupo in groupby(ordenado, key=lambda v: v["cidade"]):
    itens = list(grupo)
    print(f"   {cidade:<12} {len(itens)} venda(s), total {sum(i['valor'] for i in itens):,}")

> 💡 **Quando `groupby` é melhor que `defaultdict`?** Quando os dados **já vêm ordenados** — típico de arquivo de log ou resultado de consulta SQL com `ORDER BY`. Aí o `groupby` agrupa sem carregar tudo na memória.
>
> Se os dados estão desordenados, o `sorted()` obrigatório já materializa tudo — e aí `defaultdict` é mais simples e mais rápido.

In [ ]:
# tee: clona um iterador quando você precisa de duas passadas
numeros = iter([10, 20, 30, 40, 50])
a, b = tee(numeros, 2)

print("Soma pela cópia a:", sum(a))
print("Máximo pela cópia b:", max(b))

> ⚠️ **`tee` guarda em memória tudo que uma cópia consumiu e a outra ainda não.** Se você processar `a` inteiro antes de tocar em `b`, o `tee` terá bufferizado a coleção inteira — perdendo toda a vantagem. Só use quando as cópias avançam mais ou menos juntas.

In [ ]:
# product: substitui laços aninhados
canais = ["site", "app"]
meses = ["2026-05", "2026-06", "2026-07"]

print("Grade completa canal × mês:")
for canal, mes in product(canais, meses):
    print(f"  {canal:<6} {mes}")

## 🔧 Prática guiada — Pipeline de ETL com memória constante

Vamos construir um ETL completo que processa o CSV de 200 mil linhas usando poucos MB, com estatísticas de qualidade coletadas no caminho.

Esta é a arquitetura que você vai reencontrar, em escala, no Módulo 10.

In [ ]:
import functools
from collections import Counter, defaultdict


def contar_passagem(rotulo, contadores):
    """Decorador de gerador: conta quantos itens passam por uma etapa.

    💡 Instrumentar cada estágio permite descobrir ONDE os registros
       se perdem no pipeline. Sem isso, você só vê o total final e
       não sabe qual filtro comeu os dados.
    """
    def decorador(gerador):
        @functools.wraps(gerador)
        def envelope(*args, **kwargs):
            for item in gerador(*args, **kwargs):
                contadores[rotulo] += 1
                yield item
        return envelope
    return decorador


CONTADORES = Counter()
PROBLEMAS = Counter()


@contar_passagem("1_lidas", CONTADORES)
def extrair(caminho):
    """Extração: lê o CSV linha a linha."""
    with open(caminho, newline="", encoding="utf-8") as f:
        yield from csv.DictReader(f)


@contar_passagem("2_validas", CONTADORES)
def validar(linhas):
    """Transformação: converte tipos e descarta linhas inválidas."""
    for linha in linhas:
        try:
            quantidade = int(linha["quantidade"])
            preco = float(linha["preco"])
        except (ValueError, KeyError):
            PROBLEMAS["tipo_invalido"] += 1
            continue

        if quantidade <= 0:
            PROBLEMAS["quantidade_invalida"] += 1
            continue
        if preco < 0:
            PROBLEMAS["preco_negativo"] += 1
            continue

        yield {
            "id": int(linha["id"]),
            "cidade": linha["cidade"].strip().title(),
            "produto": linha["produto"].strip(),
            "quantidade": quantidade,
            "preco": preco,
            "status": linha["status"].strip().lower(),
        }


@contar_passagem("3_faturadas", CONTADORES)
def apenas_faturadas(registros):
    """Transformação: regra de negócio."""
    for r in registros:
        if r["status"] == "pago":
            yield r
        else:
            PROBLEMAS[f"status_{r['status']}"] += 1


@contar_passagem("4_enriquecidas", CONTADORES)
def enriquecer(registros):
    """Transformação: colunas derivadas."""
    for r in registros:
        r["total"] = round(r["quantidade"] * r["preco"], 2)
        r["faixa"] = ("alto" if r["total"] >= 5000 else
                      "medio" if r["total"] >= 1000 else "baixo")
        yield r


print("✅ Pipeline definido (nada executado ainda)")

In [ ]:
# Carga: consome o pipeline e agrega — memória constante
tracemalloc.start()
inicio = time.perf_counter()

agregado = defaultdict(lambda: {"pedidos": 0, "itens": 0, "receita": 0.0})
por_faixa = Counter()

pipeline = enriquecer(apenas_faturadas(validar(extrair(CSV))))

for registro in pipeline:
    a = agregado[registro["cidade"]]
    a["pedidos"] += 1
    a["itens"] += registro["quantidade"]
    a["receita"] += registro["total"]
    por_faixa[registro["faixa"]] += 1

tempo = time.perf_counter() - inicio
_, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"⏱  {tempo:.2f}s   💾 pico: {pico/1024/1024:.1f} MB\n")

print("FUNIL DO PIPELINE")
print("─" * 46)
anterior = None
for etapa in sorted(CONTADORES):
    n = CONTADORES[etapa]
    perda = f"  (−{anterior - n:,})" if anterior and anterior > n else ""
    print(f"  {etapa:<18}{n:>10,}{perda}")
    anterior = n

print("\nDESCARTES POR MOTIVO")
print("─" * 46)
for motivo, n in PROBLEMAS.most_common():
    print(f"  {motivo:<24}{n:>10,}")

In [ ]:
print("FATURAMENTO POR CIDADE")
print("─" * 62)
print(f"{'Cidade':<16}{'Pedidos':>10}{'Itens':>10}{'Receita':>18}")
print("─" * 62)
total_geral = sum(a["receita"] for a in agregado.values())
for cidade, a in sorted(agregado.items(), key=lambda kv: -kv[1]["receita"]):
    print(f"{cidade:<16}{a['pedidos']:>10,}{a['itens']:>10,}{a['receita']:>18,.2f}")
print("─" * 62)
print(f"{'TOTAL':<16}{'':<20}{total_geral:>18,.2f}")

print("\nDISTRIBUIÇÃO POR FAIXA")
for faixa in ["baixo", "medio", "alto"]:
    n = por_faixa[faixa]
    barra = "█" * int(40 * n / max(por_faixa.values()))
    print(f"  {faixa:<8}{n:>8,}  {barra}")

> 💭 **O que acabamos de construir:** um ETL que processa 200 mil linhas com **poucos MB** de memória, instrumentado em cada etapa, com contabilidade de descartes por motivo.
>
> Troque o arquivo por um de 4 GB e **nada muda** — nem o código, nem o consumo de memória. Só o tempo.
>
> Esse é o poder da avaliação preguiçosa, e é a razão de os geradores existirem.

## 📝 Exercícios

**E1.** Escreva um gerador `pares(n)` que produza os n primeiros números pares. Depois um `primos()` infinito e pegue os 15 primeiros com `islice`.

**E2.** Escreva um gerador `janela_movel(iteravel, tamanho)` que produza tuplas de janela deslizante: `[1,2,3,4,5]` com tamanho 3 → `(1,2,3), (2,3,4), (3,4,5)`.

**E3.** Escreva `lotes(iteravel, tamanho)` que agrupe os itens em blocos de N. Útil para inserir no banco em lotes (você usou `executemany` no M03 — este gerador alimenta ele).

**E4.** Demonstre o esgotamento: crie um `map`, consuma-o com `list()`, e depois tente iterar de novo. Explique. Depois corrija de duas formas.

**E5.** Escreva um iterador **de classe** `Paginador(itens, por_pagina)` que devolva uma página por vez e tenha também `.total_paginas` e `.pagina_atual`. Depois escreva a versão em gerador e compare.

**E6.** Escreva `ler_reverso(caminho)` que produza as linhas de um arquivo **de trás para frente**, sem carregar tudo na memória. *(Dica: `seek` do fim para o início, em blocos.)*

**E7.** Monte um pipeline de 4 etapas sobre `vendas_grande.csv` que: leia, converta, filtre produtos com preço acima de R$ 1.000 e agregue por produto. Meça a memória com `tracemalloc`.

**E8.** Use `groupby` corretamente para agrupar as vendas por produto. Depois faça o mesmo com `defaultdict` e compare tempo e memória em 200 mil linhas.

**E9.** Escreva uma corrotina `media_movel(n)` que receba valores via `send()` e devolva a média dos últimos n. *(Dica: `collections.deque(maxlen=n)`.)*

**E10.** Escreva `intercalar(*iteraveis)` que produza um item de cada iterável em rodízio, até todos acabarem. *(Dica: `zip_longest` com sentinela, ou `cycle` + remoção.)*

**E11.** Implemente `tee` do zero: uma função que receba um iterador e devolva n cópias independentes, usando `deque` como buffer.

**E12.** Escreva um decorador `@gerador_com_progresso(a_cada=10000)` que imprima o progresso a cada N itens produzidos, sem alterar o gerador decorado.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

## 📋 Cola de referência

```python
# ── Protocolo ──
it = iter(colecao)        # obtém iterador
next(it)                  # próximo item
next(it, padrao)          # com valor padrão em vez de StopIteration
# StopIteration encerra o for

# Iterável  → tem __iter__
# Iterador  → tem __iter__ E __next__, tem estado, ESGOTA
from collections.abc import Iterable, Iterator
isinstance(x, Iterator)

# ⚠️ map, filter, zip, enumerate, reversed e geradores são ITERADORES

# ── Iterador de classe ──
class Meu:
    def __iter__(self):
        self.i = 0
        return self
    def __next__(self):
        if acabou: raise StopIteration
        return valor

# ── Gerador ──
def gerador(n):
    for i in range(n):
        yield i           # devolve E congela

g = gerador(3)            # não executa nada
next(g)                   # executa até o 1º yield

def delegar(colecoes):
    for c in colecoes:
        yield from c      # ≡ for x in c: yield x

# ── Corrotina ──
def acumulador():
    total = 0
    while True:
        valor = yield total     # devolve E recebe
        total += valor or 0

acc = acumulador()
next(acc)                 # ⚠️ priming obrigatório
acc.send(100)
acc.close()               # levanta GeneratorExit lá dentro

# ── Pipeline ──
resultado = etapa4(etapa3(etapa2(etapa1(fonte))))
# nada executa até você consumir

# ── itertools ──
from itertools import *
count(10, 5)                      # infinito: 10, 15, 20...
cycle(xs)                         # repete para sempre
islice(it, 10)                    # fatia sem materializar
takewhile(pred, it)  dropwhile(pred, it)
chain(a, b, c)                    # concatena
zip_longest(a, b, fillvalue=0)
product(a, b)                     # cartesiano
combinations(xs, 2)  permutations(xs, 2)
accumulate(xs)                    # soma acumulada
pairwise(xs)                      # (3.10+) pares consecutivos
tee(it, 2)                        # ⚠️ bufferiza a diferença
groupby(sorted(xs, key=k), key=k) # ⚠️ SÓ agrupa consecutivos

# ── Medição ──
import tracemalloc, sys
tracemalloc.start()
_, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()
sys.getsizeof(objeto)
```

## ✅ Checklist de saída

- [ ] Explico o que o `for` faz por baixo (`iter`, `next`, `StopIteration`)
- [ ] Diferencio iterável de iterador e sei por que importa
- [ ] Sei que `map`/`filter`/`zip`/geradores **esgotam** ao ser consumidos
- [ ] Escrevo um iterador de classe com `__iter__`/`__next__`
- [ ] **Prefiro geradores a classes iteradoras**
- [ ] Entendo que `yield` congela e retoma a função
- [ ] Sei que chamar uma função geradora não executa nada
- [ ] Escrevo geradores infinitos e sei como pará-los
- [ ] Uso avaliação preguiçosa para processar arquivos grandes
- [ ] Monto pipelines de geradores encadeados
- [ ] Uso `yield from` para delegar, inclusive recursivamente
- [ ] Sei o que é uma corrotina e por que o priming é necessário
- [ ] Conheço `islice`, `chain`, `accumulate`, `pairwise`, `zip_longest`
- [ ] **Sei que `groupby` só agrupa consecutivos** e ordeno antes
- [ ] Sei que `tee` bufferiza e quando isso é um problema
- [ ] Meço memória com `tracemalloc`

---

### ➡️ Próxima aula

**`04_03_Orientacao_a_Objetos.ipynb`** — Classes, encapsulamento, herança vs composição e métodos dunder. Onde o Atlas deixa de ser um script e vira um sistema.